<div dir="rtl">
<h1>اجازه داریم همین یک نمونه را حفظ کنیم</h1>
<p>درس 68 از 76 · اگر حتی یک Batch یاد گرفته نشود چه کنیم؟ · <code dir="ltr">61-one-batch</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-05/61-one-batch.html">📖 بازگشت به همین درس</a></p>
<p>یک آزمون کوچک یادگیری بسازید و نبودِ Step را از نبودِ Gradient جدا کنید.</p><p>پیش‌نیاز: حلقهٔ آموزش، بررسی تغییر وزن و محدودیت آزمون تک‌نمونه‌ای.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>اگر Dropout صفر و وزن ثابت باشد، آیا تکرار forward روی همان نمونه Loss تازه‌ای می‌دهد؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
def fresh_model():
    torch.manual_seed(7)
    return MiniGPT(ModelConfig(12,8,16,2,2,0.0))
x,y = torch.tensor([[1,2,3,4,5,6]]),torch.tensor([[2,3,4,5,6,7]])
model = fresh_model()
print('same input losses:',model(x,y)[1].item(),model(x,y)[1].item())

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>overfit_history(Model,x,y,steps) یک AdamW با lr=0.01 بسازد و یک Batch ثابت را steps بار آموزش دهد. فهرست Loss پیش از آموزش و پس از هر update را برگرداند؛ طول فهرست steps+1 است.</p>
</div>

In [ ]:
def overfit_history(model, x, y, steps):
    # TODO: همان داده در هر گام، با update واقعی
    return None

In [ ]:
def test_exercise():
    trial = fresh_model()
    result = overfit_history(trial,x,y,80)
    if result is None:
        return False
    assert len(result)==81
    assert result[-1]<result[0]/10
    assert all(torch.isfinite(torch.tensor(value)) for value in result)
    other = fresh_model()
    before = other.token_embedding.weight.detach().clone()
    zero = overfit_history(other,x,y,0)
    assert len(zero)==1 and torch.equal(before,other.token_embedding.weight)
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: overfit_history')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط فعال‌بودن Dropout را عوض کنید، بدون هیچ optimizer.step. نوسان Loss را با یادگیری اشتباه نگیرید.</p>
</div>

In [ ]:
for dropout in (0.0,0.5):
    torch.manual_seed(7)
    trial = MiniGPT(ModelConfig(12,8,16,2,2,dropout)).train()
    before = trial.token_embedding.weight.detach().clone()
    print(dropout,[trial(x,y)[1].item() for _ in range(4)])
    assert torch.equal(before,trial.token_embedding.weight)

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>در نمونهٔ Scalar، backward وزن را حرکت نمی‌دهد. scalar_step(Parameter, Optimizer) یک گام برای کم‌کردن (parameter-3)**2 اجرا کند و مقدار تازهٔ وزن را بدهد.</p>
</div>

In [ ]:
w = torch.nn.Parameter(torch.tensor(0.0))
optimizer = torch.optim.SGD([w],lr=0.1)
((w-3)**2).backward()
print('gradient exists:',w.grad.item(),'but weight:',w.item())

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def scalar_step(parameter, optimizer):
    # TODO: محاسبهٔ مشتق و اعمال آن دو کار جدا هستند
    return None

In [ ]:
def test_repair():
    w = torch.nn.Parameter(torch.tensor(0.0))
    optimizer = torch.optim.SGD([w],lr=0.1)
    result = scalar_step(w,optimizer)
    if result is None:
        return False
    assert abs(result-0.6)<1e-6
    assert abs(scalar_step(w,optimizer)-1.08)<1e-6
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: scalar_step')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>مدل و تنظیم مرجع با آزمایش overfit پروژه سازگارند. این موفقیت فقط سلامت یادگیری همان نمونه را نشان می‌دهد؛ نشت داده یا اشکال Validation را رد نمی‌کند.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چرا موفقیت این آزمون اجازه نمی‌دهد بگوییم مدل روی متن تازه خوب کار می‌کند؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-09/chapter-05/61-one-batch.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/61-one-batch.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>